# Penyusunan Transkrip Acuan
### Analisis dan Perancangan Sistem · Interaksi Manusia dan Komputer · Arsitektur dan Organisasi Komputer

Notebook ini menjalankan subbab **3.3.1** (penyaringan dan pengumpulan), **3.6.1** (praproses audio), dan **3.3.2** (penyusunan transkrip acuan) sampai menghasilkan master transkrip acuan berformat TSV.

## Alur

```
[1] Manifest rekaman            -> data/output/manifest_rekaman.csv
[2] Unduh audio (yt-dlp)        -> data/raw/audio_src/
[3] Praproses audio (ffmpeg)    -> data/processed/audio/    16 kHz, mono, PCM 16-bit
[4] Transkripsi large-v3        -> data/raw/json/           arsip, tidak disunting
[5] Pemeriksaan awal rekaman    -> deteksi intro yang tersisa
[6] Prioritas simak             -> data/output/review_priority.csv
[7] Ekspor SRT                  -> data/processed/srt/
    ========== KOREKSI MANUAL DI SUBTITLE EDIT ==========
    Simpan hasil ke              data/processed/srt_done/
[8] SRT terkoreksi -> TSV       -> data/output/master/
[9] Validasi
[10] Glosarium
[11] Ringkasan untuk Bab 4
```

## 0. Persiapan

In [ ]:
# Jalankan sekali bila pustaka belum terpasang.
# !pip install faster-whisper yt-dlp pandas
# FFmpeg harus tersedia di PATH (cek: !ffmpeg -version)

In [3]:
from pathlib import Path
import json, re, subprocess, shutil, unicodedata, datetime, time
import pandas as pd

# Notebook berada di notebooks/ , sehingga akar proyek satu tingkat di atas.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW        = ROOT / "data" / "raw"
PROC       = ROOT / "data" / "processed"
OUT        = ROOT / "data" / "output"

AUDIO_SRC  = RAW  / "audio_src"      # hasil unduh, apa adanya
JSON_RAW   = RAW  / "json"           # keluaran large-v3, arsip
AUDIO_PROC = PROC / "audio"          # 16 kHz mono PCM16
SRT_TODO   = PROC / "srt"            # untuk dikoreksi
SRT_DONE   = PROC / "srt_done"       # hasil koreksi manual
MASTER     = OUT  / "master"         # TSV transkrip acuan

for d in (AUDIO_SRC, JSON_RAW, AUDIO_PROC, SRT_TODO, SRT_DONE, MASTER, OUT):
    d.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUT / "manifest_rekaman.csv"

print("Akar proyek :", ROOT)
print("FFmpeg      :", shutil.which("ffmpeg") or "TIDAK DITEMUKAN")
print("yt-dlp      :", shutil.which("yt-dlp") or "TIDAK DITEMUKAN")
print("pandas      :", pd.__version__)

Akar proyek : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative
FFmpeg      : C:\Users\Aufii\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE
yt-dlp      : c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\.venv\Scripts\yt-dlp.EXE
pandas      : 3.0.3


In [ ]:
CFG = {
    "model_acuan": "large-v3",
    "device": "cuda",
    "compute_type": "float16",
    "language": "id",
    "task": "transcribe",
    "beam_size": 5,
    "temperature": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
    "vad_filter": True,
    "condition_on_previous_text": False,
    "word_timestamps": False,
    "sample_rate": 16000,
    "channels": 1,
    "loudnorm_I": -23.0,
    "loudnorm_TP": -2.0,
    "loudnorm_LRA": 7.0,
}

RUN_ULANG = {
    "unduh": False,
    "praproses": False,
    "transkripsi": False,
    "master": False,
}
CFG

{'model_acuan': 'large-v3',
 'device': 'cuda',
 'compute_type': 'float16',
 'language': 'id',
 'task': 'transcribe',
 'beam_size': 5,
 'temperature': [0.0, 0.2, 0.4, 0.6, 0.8, 1.0],
 'vad_filter': True,
 'condition_on_previous_text': False,
 'word_timestamps': False,
 'sample_rate': 16000,
 'channels': 1,
 'loudnorm_I': -23.0,
 'loudnorm_TP': -2.0,
 'loudnorm_LRA': 7.0}

## 1. Manifest rekaman

Kolom yang **wajib diisi manual** sebelum sel [3] dijalankan:

| Kolom | Isi |
|---|---|
| `ada_takarir_manusia` | `True`/`False`. Cek di halaman video (Settings → Subtitles). Yang tertera *auto-generated* boleh, yang tertera nama bahasa tanpa keterangan itu buatan manusia, dan rekamannya harus dikeluarkan sesuai kriteria penyaringan nomor 2. |
| `lolos_penyaringan` | `True`/`False`, dengan alasan di kolom `catatan` bila `False` |
| `trim_akhir_detik` | detik penutup yang bukan materi kuliah |

`trim_awal_detik` sudah terisi dari hasil pemeriksaan: 25 detik untuk seluruh AOK, 0 untuk sisanya. Sel [5] memverifikasi apakah masih ada intro yang tersisa setelah pemotongan.

Seluruh kolom disimpan sebagai teks. Konversi ke angka dilakukan eksplisit di tempat yang membutuhkannya.

In [ ]:
REKAMAN = [
    ("APS-01", "APS", "https://www.youtube.com/watch?v=HOG8CUse5HY", "13:18",  0, "P1"),
    ("APS-02", "APS", "https://www.youtube.com/watch?v=6OU9phN5dDQ", "11:08",  0, "P2"),
    ("APS-03", "APS", "https://www.youtube.com/watch?v=k1z9gy1fVy8", "10:42",  0, "P1"),
    ("APS-04", "APS", "https://www.youtube.com/watch?v=OMliDo_CpxE", "12:19",  0, "P2"),
    ("APS-05", "APS", "https://www.youtube.com/watch?v=ToESrvlgg3o",  "9:41",  0, "P1"),
    ("APS-06", "APS", "https://www.youtube.com/watch?v=ikke0VwiQS0", "20:41",  0, "P2"),

    ("IMK-01", "IMK", "https://www.youtube.com/watch?v=VWIuwAQZWSY", "15:41",  0, ""),
    ("IMK-02", "IMK", "https://www.youtube.com/watch?v=nh-xG-v8whc", "18:42",  0, ""),
    ("IMK-03", "IMK", "https://www.youtube.com/watch?v=aS_dB3dvWIU", "15:48",  0, ""),
    ("IMK-04", "IMK", "https://www.youtube.com/watch?v=TNr8swEdr9g", "16:36",  0, ""),
    ("IMK-05", "IMK", "https://www.youtube.com/watch?v=mC_PPEjBLTs", "17:30",  0, ""),

    ("AOK-01", "AOK", "https://www.youtube.com/watch?v=SAjBsxQFV8w", "14:17", 25, ""),
    ("AOK-02", "AOK", "https://www.youtube.com/watch?v=-a9iZ8y999Y", "34:42", 25, ""),
    ("AOK-03", "AOK", "https://www.youtube.com/watch?v=6pUbgRsESkc", "12:48", 25, ""),
    ("AOK-04", "AOK", "https://www.youtube.com/watch?v=mkdzKqmMxto", "19:56", 25, ""),
    ("AOK-05", "AOK", "https://www.youtube.com/watch?v=sFFBdaQj_J0", "18:05", 25, ""),
    ("AOK-06", "AOK", "https://www.youtube.com/watch?v=6NVTqNtVrCw", "15:32", 25, ""),
    ("AOK-07", "AOK", "https://www.youtube.com/watch?v=TCAx_b8fWF0", "15:58", 25, ""),
    ("AOK-08", "AOK", "https://www.youtube.com/watch?v=JLxEKYbm_c8", "13:48", 25, ""),
]

CATATAN_AWAL = {
    "APS-02": "pengajar menyebut mata kuliah RPL; bahan kajian beririsan dengan APS "
              "dan sesuai kriteria penyaringan nomor 1",
}

NAMA_MK = {
    "APS": "Analisis dan Perancangan Sistem",
    "IMK": "Interaksi Manusia dan Komputer",
    "AOK": "Arsitektur dan Organisasi Komputer",
}

KOLOM_MANIFEST = [
    "file_id", "mata_kuliah", "nama_mata_kuliah", "url", "penutur",
    "durasi_tertera", "durasi_tertera_detik",
    "trim_awal_detik", "trim_akhir_detik",
    "durasi_asli_detik", "durasi_potong_detik",
    "ada_takarir_manusia", "lolos_penyaringan", "split",
    "tanggal_unduh", "catatan",
]

def mmss_ke_detik(s):
    b = [float(x) for x in s.split(":")]
    return b[0] * 60 + b[1] if len(b) == 2 else b[0] * 3600 + b[1] * 60 + b[2]

def manifest_baru():
    baris = []
    for fid, mk, url, dur, trim, penutur in REKAMAN:
        baris.append({
            "file_id": fid,
            "mata_kuliah": mk,
            "nama_mata_kuliah": NAMA_MK[mk],
            "url": url,
            "penutur": penutur,
            "durasi_tertera": dur,
            "durasi_tertera_detik": f"{mmss_ke_detik(dur):.3f}",
            "trim_awal_detik": str(trim),
            "trim_akhir_detik": "0",
            "durasi_asli_detik": "",
            "durasi_potong_detik": "",
            "ada_takarir_manusia": "",
            "lolos_penyaringan": "",
            "split": "",
            "tanggal_unduh": "",
            "catatan": CATATAN_AWAL.get(fid, ""),
        })
    return pd.DataFrame(baris, columns=KOLOM_MANIFEST)

def simpan_manifest(df):
    df.to_csv(MANIFEST_PATH, index=False, encoding="utf-8-sig")

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH, dtype=str, keep_default_na=False)
    print("Manifest lama dimuat, tidak ditimpa.")
    print("Hapus berkasnya lebih dahulu bila ingin dibangun ulang dari REKAMAN.")
else:
    manifest = manifest_baru()
    simpan_manifest(manifest)
    print("Manifest dibuat:", MANIFEST_PATH)

tot = manifest["durasi_tertera_detik"].astype(float).sum()
print(f"\n{len(manifest)} rekaman, total {tot/3600:.2f} jam\n")
print(manifest.assign(d=manifest["durasi_tertera_detik"].astype(float))
              .groupby("mata_kuliah")["d"]
              .agg(jumlah="count", detik="sum")
              .assign(jam=lambda x: (x["detik"]/3600).round(2))
              .to_string())

kosong = manifest[(manifest["ada_takarir_manusia"] == "") |
                  (manifest["lolos_penyaringan"] == "")]["file_id"].tolist()
if kosong:
    print(f"\nBELUM DIISI MANUAL ({len(kosong)}): {', '.join(kosong)}")

Manifest lama dimuat, tidak ditimpa.
Hapus berkasnya lebih dahulu bila ingin dibangun ulang dari REKAMAN.

19 rekaman, total 5.12 jam

             jumlah   detik   jam
mata_kuliah                      
AOK               8  8706.0  2.42
APS               6  4669.0  1.30
IMK               5  5057.0  1.40


In [ ]:
trim_baru = {fid: str(trim) for fid, _, _, _, trim, _ in REKAMAN}
ubah = 0
for i, b in manifest.iterrows():
    baru = trim_baru.get(b["file_id"])
    if baru is not None and b["trim_awal_detik"] != baru:
        print(f"{b['file_id']}: {b['trim_awal_detik']} -> {baru}")
        manifest.at[i, "trim_awal_detik"] = baru
        ubah += 1
simpan_manifest(manifest)
print(f"\n{ubah} baris diperbarui")


0 baris diperbarui


## 2. Unduh audio

Hanya jalur audio yang diunduh. Berkas video tidak diperlukan dan hanya memakan ruang.

In [7]:
def unduh(file_id, url, force=False):
    tujuan = AUDIO_SRC / f"{file_id}.m4a"
    if tujuan.exists() and not force:
        return tujuan, "sudah ada"
    perintah = [
        "yt-dlp", "-f", "bestaudio[ext=m4a]/bestaudio",
        "--extract-audio", "--audio-format", "m4a", "--audio-quality", "0",
        "--no-playlist", "--no-write-subs", "--no-write-auto-subs",
        "-o", str(AUDIO_SRC / f"{file_id}.%(ext)s"),
        url,
    ]
    hasil = subprocess.run(perintah, capture_output=True, text=True)
    if hasil.returncode != 0:
        pesan = hasil.stderr.strip().splitlines()[-1] if hasil.stderr else "gagal"
        return None, pesan[:120]
    return tujuan, "terunduh"


hari_ini = datetime.date.today().isoformat()
for i, baris in manifest.iterrows():
    berkas, status = unduh(baris["file_id"], baris["url"], RUN_ULANG["unduh"])
    print(f"{baris['file_id']:8s} {status}")
    if berkas is not None and not manifest.at[i, "tanggal_unduh"].strip():
        manifest.at[i, "tanggal_unduh"] = hari_ini

simpan_manifest(manifest)

APS-01   sudah ada
APS-02   sudah ada
APS-03   sudah ada
APS-04   sudah ada
APS-05   sudah ada
APS-06   sudah ada
IMK-01   sudah ada
IMK-02   sudah ada
IMK-03   sudah ada
IMK-04   sudah ada
IMK-05   sudah ada
AOK-01   sudah ada
AOK-02   sudah ada
AOK-03   sudah ada
AOK-04   sudah ada
AOK-05   sudah ada
AOK-06   sudah ada
AOK-07   sudah ada
AOK-08   sudah ada


## 3. Praproses audio (subbab 3.6.1)

Penyeragaman ke 16 kHz / mono / PCM 16-bit mengikuti laju cuplik masukan yang dipelajari Whisper, normalisasi lantang EBU R128, dan pemotongan pembuka serta penutup yang bukan materi kuliah.

Penyaring VAD **tidak** dijalankan di sini.

In [ ]:
def durasi_detik(berkas):
    hasil = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(berkas)],
        capture_output=True, text=True)
    try:
        return float(hasil.stdout.strip())
    except ValueError:
        return None


def praproses(file_id, trim_awal=0.0, trim_akhir=0.0, force=False):
    sumber = AUDIO_SRC / f"{file_id}.m4a"
    tujuan = AUDIO_PROC / f"{file_id}.wav"
    if not sumber.exists():
        return None, None, "sumber tidak ada"

    d_asli = durasi_detik(sumber)
    if tujuan.exists() and not force:
        return d_asli, durasi_detik(tujuan), "sudah ada"

    perintah = ["ffmpeg", "-y", "-hide_banner", "-loglevel", "error"]
    if trim_awal > 0:
        perintah += ["-ss", str(trim_awal)]
    perintah += ["-i", str(sumber)]
    if trim_akhir > 0 and d_asli is not None:
        perintah += ["-t", str(max(0.0, d_asli - trim_awal - trim_akhir))]
    perintah += [
        "-af", f"loudnorm=I={CFG['loudnorm_I']}:TP={CFG['loudnorm_TP']}:LRA={CFG['loudnorm_LRA']}",
        "-ar", str(CFG["sample_rate"]),
        "-ac", str(CFG["channels"]),
        "-c:a", "pcm_s16le",
        "-map_metadata", "-1",
        str(tujuan),
    ]
    hasil = subprocess.run(perintah, capture_output=True, text=True)
    if hasil.returncode != 0:
        return d_asli, None, "GAGAL: " + hasil.stderr.strip()[:120]
    return d_asli, durasi_detik(tujuan), "diproses"


for i, baris in manifest.iterrows():
    ta = float(baris["trim_awal_detik"] or 0)
    tk = float(baris["trim_akhir_detik"] or 0)
    d_asli, d_potong, status = praproses(baris["file_id"], ta, tk, RUN_ULANG["praproses"])

    if d_asli   is not None: manifest.at[i, "durasi_asli_detik"]   = f"{d_asli:.3f}"
    if d_potong is not None: manifest.at[i, "durasi_potong_detik"] = f"{d_potong:.3f}"

    ket_a = f"{d_asli:8.1f}s" if d_asli   is not None else "     ?  "
    ket_p = f"{d_potong:8.1f}s" if d_potong is not None else "     ?  "
    buang = f"(-{ta:.0f}s)" if ta else ""
    print(f"{baris['file_id']:8s} {status:12s} {ket_a} -> {ket_p} {buang}")

simpan_manifest(manifest)

APS-01   sudah ada       798.6s ->    798.6s 
APS-02   sudah ada       668.4s ->    668.4s 
APS-03   sudah ada       642.4s ->    642.4s 
APS-04   sudah ada       739.8s ->    739.8s 
APS-05   sudah ada       581.5s ->    581.5s 
APS-06   sudah ada      1213.8s ->   1213.8s 
IMK-01   sudah ada       941.5s ->    941.5s 
IMK-02   sudah ada      1122.5s ->   1122.5s 
IMK-03   sudah ada       948.7s ->    948.7s 
IMK-04   sudah ada       996.4s ->    996.4s 
IMK-05   sudah ada      1050.2s ->   1050.2s 
AOK-01   sudah ada       857.2s ->    832.2s (-25s)
AOK-02   sudah ada      2082.6s ->   2057.6s (-25s)
AOK-03   sudah ada       768.7s ->    743.7s (-25s)
AOK-04   sudah ada      1197.0s ->   1172.0s (-25s)
AOK-05   sudah ada      1085.8s ->   1060.8s (-25s)
AOK-06   sudah ada       932.3s ->    907.3s (-25s)
AOK-07   sudah ada       958.1s ->    933.1s (-25s)
AOK-08   sudah ada       828.9s ->    803.9s (-25s)


In [ ]:
periksa = []
for w in sorted(AUDIO_PROC.glob("*.wav")):
    h = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a:0",
         "-show_entries", "stream=sample_rate,channels,sample_fmt,codec_name",
         "-of", "json", str(w)], capture_output=True, text=True)
    s = json.loads(h.stdout)["streams"][0]
    periksa.append({"file_id": w.stem, **s})

df_periksa = pd.DataFrame(periksa)
print(df_periksa.to_string(index=False))
seragam = df_periksa[["sample_rate", "channels", "sample_fmt"]].drop_duplicates().shape[0] == 1
print("\nSeragam:", seragam)
assert seragam, "Format audio tidak seragam; periksa berkas yang menyimpang."

file_id codec_name sample_fmt sample_rate  channels
 AOK-01  pcm_s16le        s16       16000         1
 AOK-02  pcm_s16le        s16       16000         1
 AOK-03  pcm_s16le        s16       16000         1
 AOK-04  pcm_s16le        s16       16000         1
 AOK-05  pcm_s16le        s16       16000         1
 AOK-06  pcm_s16le        s16       16000         1
 AOK-07  pcm_s16le        s16       16000         1
 AOK-08  pcm_s16le        s16       16000         1
 APS-01  pcm_s16le        s16       16000         1
 APS-02  pcm_s16le        s16       16000         1
 APS-03  pcm_s16le        s16       16000         1
 APS-04  pcm_s16le        s16       16000         1
 APS-05  pcm_s16le        s16       16000         1
 APS-06  pcm_s16le        s16       16000         1
 IMK-01  pcm_s16le        s16       16000         1
 IMK-02  pcm_s16le        s16       16000         1
 IMK-03  pcm_s16le        s16       16000         1
 IMK-04  pcm_s16le        s16       16000         1
 IMK-05  pcm

### Sel pembersih (opsional)

Menghapus hasil praproses, transkripsi, dan SRT agar dapat dibangun ulang dari awal. Manifest, glosarium, dan `srt_done` **tidak** ikut terhapus.

Jalankan ini bila trim berubah setelah transkripsi sempat dikerjakan. Ubah `BERSIHKAN` menjadi `True` lebih dahulu.

In [10]:
BERSIHKAN = False
if BERSIHKAN:
    for folder, pola in ((AUDIO_PROC, "*.wav"), (JSON_RAW, "*.json"), (SRT_TODO, "*.srt")):
        n = 0
        for f in folder.glob(pola):
            f.unlink(); n += 1
        print(f"{folder.name}: {n} berkas dihapus")
    manifest["durasi_potong_detik"] = ""
    simpan_manifest(manifest)
    print("\nJalankan ulang sel [3] dan [4].")
else:
    print("BERSIHKAN masih False. Tidak ada yang dihapus.")

BERSIHKAN masih False. Tidak ada yang dihapus.


## 4. Transkripsi awal dengan large-v3 (subbab 3.3.2)

Keluaran disimpan sebagai JSON dan **tidak pernah disunting**. Berkas ini menjadi jejak audit sekaligus sumber kolom `teks_asr` pada master, yang memungkinkan pelaporan proporsi transkrip yang dikoreksi. Selain teks dan stempel waktu, tiga besaran ikut disimpan: `avg_logprob`, `no_speech_prob`, dan `compression_ratio`.

In [46]:
from faster_whisper import WhisperModel

model = WhisperModel(CFG["model_acuan"], device=CFG["device"], compute_type=CFG["compute_type"])
print("Model dimuat:", CFG["model_acuan"], "|", CFG["device"], CFG["compute_type"])

Model dimuat: large-v3 | cuda float16


In [47]:
def transkripsi(file_id, force=False):
    audio  = AUDIO_PROC / f"{file_id}.wav"
    tujuan = JSON_RAW / f"{file_id}.json"
    if tujuan.exists() and not force:
        return "sudah ada"
    if not audio.exists():
        return "audio tidak ada"

    t0 = time.perf_counter()
    segmen_iter, info = model.transcribe(
        str(audio),
        language=CFG["language"],
        task=CFG["task"],
        beam_size=CFG["beam_size"],
        temperature=CFG["temperature"],
        vad_filter=CFG["vad_filter"],
        condition_on_previous_text=CFG["condition_on_previous_text"],
        word_timestamps=CFG["word_timestamps"],
    )
    segmen = [{
        "id": s.id, "start": round(s.start, 3), "end": round(s.end, 3),
        "text": s.text.strip(),
        "avg_logprob": s.avg_logprob,
        "no_speech_prob": s.no_speech_prob,
        "compression_ratio": s.compression_ratio,
        "temperature": s.temperature,
    } for s in segmen_iter]
    lama = time.perf_counter() - t0

    tujuan.write_text(json.dumps({
        "file_id": file_id,
        "model": CFG["model_acuan"],
        "compute_type": CFG["compute_type"],
        "parameter": {k: CFG[k] for k in
                      ("language", "task", "beam_size", "temperature",
                       "vad_filter", "condition_on_previous_text", "word_timestamps")},
        "durasi_audio": info.duration,
        "durasi_setelah_vad": getattr(info, "duration_after_vad", None),
        "waktu_proses": round(lama, 3),
        "jumlah_segmen": len(segmen),
        "segments": segmen,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    rtf = lama / info.duration if info.duration else float("nan")
    return f"{len(segmen):4d} segmen, {lama:6.0f}s, RTF~{rtf:.3f}"


for fid in manifest["file_id"]:
    print(f"{fid:8s} {transkripsi(fid, RUN_ULANG['transkripsi'])}")

APS-01    165 segmen,    245s, RTF~0.307
APS-02    128 segmen,    293s, RTF~0.439
APS-03    122 segmen,    118s, RTF~0.183
APS-04    172 segmen,    137s, RTF~0.185
APS-05    231 segmen,     88s, RTF~0.152
APS-06    218 segmen,    228s, RTF~0.188
IMK-01    190 segmen,    384s, RTF~0.408
IMK-02    207 segmen,    451s, RTF~0.402
IMK-03    171 segmen,    385s, RTF~0.405
IMK-04    170 segmen,    398s, RTF~0.400
IMK-05    372 segmen,    492s, RTF~0.469
AOK-01    378 segmen,    303s, RTF~0.364
AOK-02    650 segmen,    615s, RTF~0.299
AOK-03    170 segmen,    239s, RTF~0.322
AOK-04    484 segmen,    389s, RTF~0.332
AOK-05    376 segmen,    339s, RTF~0.320
AOK-06    391 segmen,    313s, RTF~0.345
AOK-07    291 segmen,    314s, RTF~0.337
AOK-08    315 segmen,    286s, RTF~0.356


In [ ]:
rtf = []
for jf in sorted(JSON_RAW.glob("*.json")):
    d = json.loads(jf.read_text(encoding="utf-8"))
    if d.get("durasi_audio"):
        rtf.append({"file_id": d["file_id"],
                    "durasi_audio": round(d["durasi_audio"], 1),
                    "waktu_proses": round(d["waktu_proses"], 1),
                    "rtf": round(d["waktu_proses"] / d["durasi_audio"], 3)})
if rtf:
    df_rtf = pd.DataFrame(rtf)
    print(df_rtf.to_string(index=False))
    print(f"\nRTF rerata large-v3: {df_rtf['rtf'].mean():.3f} "
          f"(min {df_rtf['rtf'].min():.3f}, maks {df_rtf['rtf'].max():.3f})")

file_id  durasi_audio  waktu_proses   rtf
 AOK-01         832.2         302.7 0.364
 AOK-02        2057.6         615.3 0.299
 AOK-03         743.7         239.3 0.322
 AOK-04        1172.0         389.4 0.332
 AOK-05        1060.8         339.0 0.320
 AOK-06         907.3         313.0 0.345
 AOK-07         933.1         314.0 0.337
 AOK-08         803.9         285.8 0.356
 APS-01         798.6         245.5 0.307
 APS-02         668.4         293.4 0.439
 APS-03         642.4         117.7 0.183
 APS-04         739.8         136.7 0.185
 APS-05         581.5          88.2 0.152
 APS-06        1213.8         227.6 0.188
 IMK-01         941.5         384.1 0.408
 IMK-02        1122.5         451.2 0.402
 IMK-03         948.7         384.6 0.405
 IMK-04         996.4         398.1 0.400
 IMK-05        1050.2         492.4 0.469

RTF rerata large-v3: 0.327 (min 0.152, maks 0.469)


## 5. Pemeriksaan awal rekaman

Segmen pertama yang durasinya janggal panjang menandakan masih ada bagian bukan-ujaran di depan. Gejalanya: VAD menangkap musik intro sehingga segmen dibentangkan mundur, dan teks ujaran pertama menerima stempel waktu jauh lebih awal daripada ujaran sebenarnya.

In [ ]:
def segmen_awal(file_id, n=4):
    jf = JSON_RAW / f"{file_id}.json"
    if not jf.exists():
        return None
    return json.loads(jf.read_text(encoding="utf-8"))["segments"][:n]


AMBANG_INTRO = 10.0

tertandai = []
for fid in manifest["file_id"]:
    seg = segmen_awal(fid)
    if not seg:
        continue
    s0 = seg[0]
    dur = s0["end"] - s0["start"]
    janggal = dur > AMBANG_INTRO
    if janggal:
        tertandai.append(fid)
    tanda = "  <-- PERIKSA" if janggal else ""
    print(f"\n{fid}{tanda}")
    for i, s in enumerate(seg, 1):
        print(f"   [{i}] {s['start']:7.2f}-{s['end']:7.2f} "
              f"({s['end']-s['start']:5.2f}s)  {s['text'][:78]}")

print("\n" + "=" * 60)
if tertandai:
    print(f"Masih tertandai ({len(tertandai)}): {', '.join(tertandai)}")
    print("Segmen pertama yang panjang belum tentu intro. Putar dari detik 0 dan")
    print("pastikan: bila ujaran memang dimulai di situ, biarkan apa adanya.")
else:
    print("Tidak ada segmen pembuka yang janggal.")


APS-01
   [1]    2.00-  10.26 ( 8.26s)  Halo semuanya, saya Fida Mastrika dan dalam video tentang analisis dan desain 
   [2]   10.26-  15.22 ( 4.96s)  apa yang dimaksud dengan analisis dan desain sistem.
   [3]   15.84-  24.60 ( 8.76s)  Dalam dua video sebelumnya, saya sudah menjelaskan tentang apa itu sistem dan 
   [4]   24.60-  32.48 ( 7.88s)  Kalau kalian belum nonton videonya, silakan klik link dari video-video tersebu

APS-02
   [1]    0.00-   3.58 ( 3.58s)  Selamat datang di video materi mata kuliah rekayasa perangkat lunak.
   [2]    4.12-   8.16 ( 4.04s)  Pada kesempatan kali ini, materi yang dibahas adalah tentang konsep perancanga
   [3]    8.66-   9.40 ( 0.74s)  Selamat menyimak.
   [4]   10.34-  12.56 ( 2.22s)  Topik pembahasan dalam video ini ada tiga.

APS-03
   [1]    0.43-   2.69 ( 2.26s)  Halo semuanya, saya Fida Mastrika.
   [2]    2.91-   6.33 ( 3.42s)  Dalam video tentang analisis dan design system kali ini,
   [3]    6.51-  14.21 ( 7.70s)  saya akan menjelaskan 

## 6. Prioritas simak

Segmen dengan rasio kompresi tinggi, peluang tanpa-wicara tinggi, atau log-probabilitas rendah adalah kandidat halusinasi dan pengulangan. Koenecke, et al. (2024) mengaitkan durasi non-vokal panjang dengan munculnya halusinasi, sehingga jeda panjang antarsegmen ikut ditandai.

Daftar ini **bukan pengganti simak penuh**. Pedoman 3.3.2 tetap menuntut seluruh audio disimak; daftar ini hanya menandai baris yang perlu perhatian ekstra.

In [50]:
AMBANG = {
    "compression_ratio": 2.4,   # di atas ini: teks berulang
    "no_speech_prob": 0.60,     # di atas ini: kemungkinan tanpa wicara
    "avg_logprob": -1.00,       # di bawah ini: model tidak yakin
    "jeda_detik": 3.0,          # jeda sebelum segmen ini
    "durasi_segmen": 28.0,      # mendekati batas 30 detik
}

def ngram_berulang(teks, n=4):
    kata = teks.lower().split()
    if len(kata) < n * 2:
        return False
    gram = [" ".join(kata[i:i+n]) for i in range(len(kata) - n + 1)]
    return len(gram) - len(set(gram)) >= 2


baris_prioritas = []
for jf in sorted(JSON_RAW.glob("*.json")):
    d = json.loads(jf.read_text(encoding="utf-8"))
    prev_end = 0.0
    for s in d["segments"]:
        alasan = []
        if s["compression_ratio"] > AMBANG["compression_ratio"]: alasan.append("kompresi-tinggi")
        if s["no_speech_prob"]   > AMBANG["no_speech_prob"]:     alasan.append("mungkin-senyap")
        if s["avg_logprob"]      < AMBANG["avg_logprob"]:        alasan.append("logprob-rendah")
        if s["start"] - prev_end > AMBANG["jeda_detik"]:         alasan.append("jeda-panjang")
        if s["end"] - s["start"] > AMBANG["durasi_segmen"]:      alasan.append("segmen-panjang")
        if ngram_berulang(s["text"]):                            alasan.append("frasa-berulang")
        if alasan:
            baris_prioritas.append({
                "file_id": d["file_id"], "seg_id": s["id"],
                "start": s["start"], "end": s["end"],
                "waktu": f"{int(s['start']//60):02d}:{s['start']%60:05.2f}",
                "alasan": "|".join(alasan),
                "compression_ratio": round(s["compression_ratio"], 2),
                "no_speech_prob": round(s["no_speech_prob"], 3),
                "avg_logprob": round(s["avg_logprob"], 3),
                "teks_asr": s["text"],
                "sudah_disimak": "",
            })

df_prioritas = pd.DataFrame(baris_prioritas)
df_prioritas.to_csv(OUT / "review_priority.csv", index=False, encoding="utf-8-sig")
print(f"{len(df_prioritas)} segmen ditandai")
if len(df_prioritas):
    print("\nAlasan:")
    print(df_prioritas["alasan"].str.split("|").explode().value_counts().to_string())
    print("\nPer rekaman:")
    print(df_prioritas["file_id"].value_counts().to_string())

5184 segmen ditandai

Alasan:
alasan
jeda-panjang      5184
mungkin-senyap       3
segmen-panjang       3

Per rekaman:
file_id
AOK-02    649
AOK-04    484
AOK-06    389
AOK-01    376
AOK-05    376
IMK-05    371
AOK-08    315
AOK-07    290
APS-05    230
APS-06    217
IMK-02    206
IMK-01    189
APS-04    172
AOK-03    170
IMK-03    170
IMK-04    169
APS-01    164
APS-02    127
APS-03    120


## 7. Ekspor SRT untuk koreksi manual

Berkas SRT ditulis ke `data/processed/srt/`. **Jangan menyunting berkas di folder itu.** Buka di Subtitle Edit, koreksi, lalu simpan ke `data/processed/srt_done/` dengan nama yang sama. Pemisahan folder ini menjaga keluaran mentah tetap utuh sebagai pembanding.

In [51]:
def detik_ke_srt(t):
    ms = int(round(t * 1000))
    j, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    d, ms = divmod(ms, 1000)
    return f"{j:02d}:{m:02d}:{d:02d},{ms:03d}"


for jf in sorted(JSON_RAW.glob("*.json")):
    d = json.loads(jf.read_text(encoding="utf-8"))
    blok = []
    for i, s in enumerate(d["segments"], start=1):
        blok.append(f"{i}\n{detik_ke_srt(s['start'])} --> {detik_ke_srt(s['end'])}\n{s['text']}\n")
    (SRT_TODO / f"{d['file_id']}.srt").write_text("\n".join(blok), encoding="utf-8")
    print(f"{d['file_id']:8s} {len(d['segments']):4d} baris")

print("\nSRT siap dikoreksi di:", SRT_TODO)

AOK-01    378 baris
AOK-02    650 baris
AOK-03    170 baris
AOK-04    484 baris
AOK-05    376 baris
AOK-06    391 baris
AOK-07    291 baris
AOK-08    315 baris
APS-01    165 baris
APS-02    128 baris
APS-03    122 baris
APS-04    172 baris
APS-05    231 baris
APS-06    218 baris
IMK-01    190 baris
IMK-02    207 baris
IMK-03    171 baris
IMK-04    170 baris
IMK-05    372 baris

SRT siap dikoreksi di: c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\processed\srt


---

# ⛔ JEDA: koreksi manual

Sel di bawah baru dijalankan setelah seluruh SRT dikoreksi.

## Penyiapan Subtitle Edit 5.2.0

Subtitle Edit memerlukan **.NET Desktop Runtime** dan mengunduh **mpv** sendiri saat pertama memutar audio. Terima keduanya.

1. `Video → Open video file`, pilih WAV dari `data/processed/audio/` — bukan m4a dari `audio_src`, agar stempel waktunya sejalan dengan berkas yang nanti ditranskripsi model uji.
2. `File → Open`, pilih SRT dari `data/processed/srt/`.
3. `Options → Settings → General`:
   - centang **Auto-convert encoding to UTF-8 on open**
   - centang **Auto-trim white-space**
   - kosongkan **Limit number of lines in subtitle text box**
   - biarkan *Default encoding* pada **UTF-8 with BOM**; parser di sel [8] memakai `utf-8-sig`
   - biarkan *Force CR+LF on save* kosong dan *Auto-backup* aktif
4. `Options → Settings → Waveform/spectrogram`:
   - kosongkan **Snap to shot changes**. Sumbernya audio kuliah, tidak ada perubahan potongan gambar, sehingga fitur itu hanya menggeser batas segmen dari batas ujaran.
   - biarkan *Allow overlap* dan *Snap to frames* kosong

Versi 5.2.0 tidak memiliki opsi *Fix common errors on load*; perkakas itu hanya berjalan bila dibuka sendiri dari menu. **Jangan pernah membuka `Tools → Fix common errors`, dan jangan menjalankan Spell check.** Keduanya mengubah kapitalisasi, tanda baca, dan ejaan, sehingga melanggar pedoman verbatim.

Bila panel waveform tetap kosong, lanjutkan saja tanpa waveform. Audio sudah dapat diputar, dan jeda panjang sudah tercatat di `review_priority.csv` dengan alasan `jeda-panjang`.

Pintasan: `Ctrl+Space` putar baris terpilih, `Alt+↓` baris berikutnya, `Ctrl+Shift+L` pecah baris pada posisi kursor.

## Pedoman transkripsi (subbab 3.3.2)

1. **Verbatim**, tanda baca dan kapitalisasi tetap dituliskan.
2. **Istilah** dicatat dalam bentuk yang benar-benar diucapkan, dengan ejaan baku bahasa asalnya bila istilah asing. Bila bentuk Inggris dan pelafalan Indonesianya tidak terbedakan secara akustik, ikuti bentuk tertulis pada materi ajar.
3. **Bilangan** ditulis sebagai kata sebagaimana diucapkan — `dua puluh`, bukan `20`. Pengubahan ke angka dikerjakan pada tahap normalisasi (3.6.3). Akronim ditulis rapat berhuruf kapital: `UML`, `API`.
4. **Jeda terisi** ditulis apa adanya (`eee`, `emm`). Kata terpotong ditulis sampai bagian yang terdengar lalu diakhiri tanda hubung: `komu-`.
5. **Tanda hubung dan bentuk ulang** mengikuti Tabel 3.4: `kata-kata`, `pandang-dengar`, `di-encode` dipertahankan sebagai satu kata.

Dua jenis kesalahan yang wajib dikejar sambil menyimak, karena tidak terbaca dari teks saja:

- **Penghapusan kata berulang** — Whisper cenderung membuang pengulangan yang benar-benar diucapkan pengajar.
- **Halusinasi** — teks yang tidak ada dalam audio, dan justru terbaca wajar. Kandidatnya ada di `data/output/review_priority.csv`.

Peringatan berwarna merah pada kolom durasi adalah batas chars/sec bawaan dunia takarir film. Abaikan; transkrip verbatim untuk penelitian memang padat dan tidak terikat aturan kecepatan baca.

## Batas segmen

Batas boleh dipecah atau digabung bila salah tempat. Sel [8] mencocokkan teks acuan ke teks ASR berdasarkan **irisan waktu**, bukan nomor baris, sehingga perubahan struktur segmen aman.

Yang harus dijaga: stempel waktu tetap urut, tidak tumpang tindih, dan tidak ada baris kosong.

---

## 8. SRT terkoreksi → master TSV

Format TSV dipilih, bukan CSV. Transkrip verbatim penuh koma dan itu adalah pemisah CSV; tab tidak pernah muncul dalam ujaran, sehingga ambiguitasnya nol. Excel membaca TSV sama baiknya lewat `Data → From Text/CSV`.

In [11]:
def baca_srt(path):
    isi = path.read_text(encoding="utf-8-sig").replace("\r\n", "\n").strip()
    pola = re.compile(
        r"(\d+)\s*\n(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*-->\s*(\d{2}:\d{2}:\d{2}[,.]\d{3})\s*\n(.*?)(?=\n\s*\n|\Z)",
        re.S)

    def ke_detik(t):
        t = t.replace(",", ".")
        j, m, d = t.split(":")
        return int(j) * 3600 + int(m) * 60 + float(d)

    hasil = []
    for m_ in pola.finditer(isi):
        teks = re.sub(r"\s*\n\s*", " ", m_.group(4)).strip()
        teks = re.sub(r"<[^>]+>", "", teks)                 # buang tag gaya
        teks = unicodedata.normalize("NFC", teks)
        teks = teks.replace("\t", " ")
        teks = re.sub(r" {2,}", " ", teks).strip()
        if teks:
            hasil.append({"start": ke_detik(m_.group(2)),
                          "end": ke_detik(m_.group(3)),
                          "teks": teks})
    return hasil


def asr_beririsan(segmen_asr, awal, akhir, min_iris=0.20):
    "Gabungkan teks ASR mentah yang beririsan waktu dengan segmen acuan."
    potong = []
    for s in segmen_asr:
        iris = min(akhir, s["end"]) - max(awal, s["start"])
        if iris > 0 and iris / max(1e-9, s["end"] - s["start"]) >= min_iris:
            potong.append(s["text"])
    return " ".join(potong).strip()

In [14]:
def bangun_master(file_id, force=False):
    srt_path  = SRT_DONE / f"{file_id}.srt"
    json_path = JSON_RAW / f"{file_id}.json"
    tsv_path  = MASTER / f"{file_id}.tsv"

    if not srt_path.exists():
        return None, "SRT terkoreksi belum ada"
    if tsv_path.exists() and not force:
        return pd.read_csv(tsv_path, sep="\t", dtype=str, keep_default_na=False), "sudah ada"

    acuan = baca_srt(srt_path)
    asr   = json.loads(json_path.read_text(encoding="utf-8"))["segments"]
    row   = manifest.loc[manifest["file_id"] == file_id].iloc[0]

    baris = [{
        "file_id": file_id,
        "mata_kuliah": row["mata_kuliah"],
        "penutur": row["penutur"],
        "seg_id": i,
        "start": f"{a['start']:.3f}",
        "end":   f"{a['end']:.3f}",
        "dur":   f"{a['end'] - a['start']:.3f}",
        "teks_asr": asr_beririsan(asr, a["start"], a["end"]),
        "teks_acuan": a["teks"],
        "flag": "",
        "catatan": "",
    } for i, a in enumerate(acuan, start=1)]

    df = pd.DataFrame(baris)
    df.to_csv(tsv_path, sep="\t", index=False, encoding="utf-8-sig")
    return df, f"{len(df)} segmen"


semua = []
for fid in manifest["file_id"]:
    df, status = bangun_master(fid, RUN_ULANG["master"])
    print(f"{fid:8s} {status}")
    if df is not None:
        semua.append(df)

master = pd.concat(semua, ignore_index=True) if semua else None
if master is not None:
    master.to_csv(OUT / "transkrip_acuan_gabungan.tsv", sep="\t",
                  index=False, encoding="utf-8-sig")
    print(f"\nGabungan: {len(master)} segmen dari {master['file_id'].nunique()} rekaman")
else:
    print("\nBelum ada SRT terkoreksi di", SRT_DONE)

APS-01   165 segmen
APS-02   128 segmen
APS-03   122 segmen
APS-04   172 segmen
APS-05   231 segmen
APS-06   218 segmen
IMK-01   190 segmen
IMK-02   207 segmen
IMK-03   171 segmen
IMK-04   170 segmen
IMK-05   372 segmen
AOK-01   378 segmen
AOK-02   650 segmen
AOK-03   170 segmen
AOK-04   484 segmen
AOK-05   376 segmen
AOK-06   390 segmen
AOK-07   291 segmen
AOK-08   314 segmen

Gabungan: 5199 segmen dari 19 rekaman


In [16]:
if master is not None:
    xlsx_path = OUT / "transkrip_acuan.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as w:
        master.to_excel(w, sheet_name="GABUNGAN", index=False)
        for fid, g in master.groupby("file_id", sort=True):
            g.to_excel(w, sheet_name=fid[:31], index=False)

        lebar = {"file_id": 10, "mata_kuliah": 12, "penutur": 9, "seg_id": 8,
                 "start": 10, "end": 10, "dur": 9,
                 "teks_asr": 60, "teks_acuan": 60, "flag": 14, "catatan": 30}
        for ws in w.book.worksheets:
            for idx, kolom in enumerate(master.columns, start=1):
                ws.column_dimensions[ws.cell(1, idx).column_letter].width = lebar.get(kolom, 14)
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
    print("Ditulis:", xlsx_path, f"({1 + master['file_id'].nunique()} sheet)")

Ditulis: c:\Users\Aufii\Documents\Informatika\OTHERS\8. RISET-SKRIPSI\assistive-communicative\data\output\transkrip_acuan.xlsx (20 sheet)


## 9. Validasi

Pemeriksaan struktural pada master. Seluruhnya harus lolos sebelum transkrip acuan dipakai.

Kode `ANGKA` dan `HUBUNG_MENGGANTUNG` bersifat peringatan, bukan kesalahan mutlak: tanda hubung menggantung memang disengaja pada kata terpotong seperti `komu-`.

In [18]:
def validasi(df):
    masalah = []
    def catat(kode, sub, pesan):
        for _, r in sub.iterrows():
            masalah.append({"file_id": r["file_id"], "seg_id": r["seg_id"],
                            "kode": kode, "pesan": pesan,
                            "cuplikan": str(r["teks_acuan"])[:70]})

    d = df.copy()
    d["start_f"] = d["start"].astype(float)
    d["end_f"]   = d["end"].astype(float)
    d["dur_f"]   = d["end_f"] - d["start_f"]
    t = d["teks_acuan"].astype(str)

    catat("KOSONG",  d[t.str.strip() == ""], "teks acuan kosong")
    catat("DUR<=0",  d[d["dur_f"] <= 0], "durasi nol atau negatif")
    catat("DUR>30",  d[d["dur_f"] > 30.0], "melebihi batas masukan Whisper 30 detik")
    catat("TAB",     d[t.str.contains("\t")], "mengandung tab")
    catat("SPASI_GANDA", d[t.str.contains("  ")], "spasi ganda")
    catat("KUTIP_TIPOGRAFIS", d[t.str.contains("[\u2018\u2019\u201c\u201d]")],
          "tanda kutip melengkung, seragamkan ke lurus")
    catat("ANGKA", d[t.str.contains(r"\d")],
          "mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata")
    catat("HUBUNG_MENGGANTUNG", d[t.str.contains(r"(?:^|\s)-|-(?:\s|$)")],
          "tanda hubung menggantung (wajar bila kata terpotong)")

    for fid, g in d.groupby("file_id", sort=False):
        g = g.sort_values("seg_id")
        tumpang = g[g["start_f"].values < g["end_f"].shift(1).fillna(-1).values]
        catat("TUMPANG_TINDIH", tumpang, "start lebih awal daripada end segmen sebelumnya")
        if not g["start_f"].is_monotonic_increasing:
            masalah.append({"file_id": fid, "seg_id": "-", "kode": "URUTAN",
                            "pesan": "stempel waktu tidak menaik", "cuplikan": ""})
    return pd.DataFrame(masalah)


if master is not None:
    lap = validasi(master)
    lap.to_csv(OUT / "laporan_validasi.csv", index=False, encoding="utf-8-sig")
    if len(lap):
        print(lap["kode"].value_counts().to_string())
        berat = lap[~lap["kode"].isin(["ANGKA", "HUBUNG_MENGGANTUNG"])]
        print(f"\nPerlu dibetulkan: {len(berat)} | peringatan saja: {len(lap) - len(berat)}")
        print("\nContoh:")
        print(lap.head(15).to_string(index=False))
    else:
        print("Seluruh pemeriksaan lolos.")

kode
ANGKA    172

Perlu dibetulkan: 0 | peringatan saja: 172

Contoh:
file_id  seg_id  kode                                                       pesan                                                               cuplikan
 APS-01      40 ANGKA mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata Ini ada 5 tahapan yang paling pokok yang ada di setiap cara untuk memb
 APS-01      41 ANGKA mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata Kalau kalian cari di internet mungkin ada yang tahapannya lebih dari 5
 APS-01      42 ANGKA mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata            Nah itu bisa saja, tapi yang 5 ini adalah yang paling pokok
 APS-02      46 ANGKA mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata Rancangan atau desain yang berkualitas haruslah memiliki sekurang-kura
 APS-02      96 ANGKA mengandung angka; pedoman 3.3.2 poin 3 menuntut bentuk kata Ada 4 dimensi perancangan di sini, yaitu arsitektur, interface, kompon
 APS-02    

In [ ]:
if master is not None:
    import difflib
    ringkas = []
    for fid, g in master.groupby("file_id", sort=False):
        a = " ".join(g["teks_asr"].astype(str)).split()
        b = " ".join(g["teks_acuan"].astype(str)).split()
        sm = difflib.SequenceMatcher(None, a, b, autojunk=False)
        sama = sum(blok.size for blok in sm.get_matching_blocks())
        ringkas.append({
            "file_id": fid,
            "kata_asr": len(a),
            "kata_acuan": len(b),
            "kata_diubah": len(b) - sama,
            "persen_diubah": round(100 * (len(b) - sama) / max(1, len(b)), 2),
        })
    df_koreksi = pd.DataFrame(ringkas)
    df_koreksi.to_csv(OUT / "ringkasan_koreksi.csv", index=False, encoding="utf-8-sig")
    print(df_koreksi.to_string(index=False))
    print(f"\nRerata proporsi kata yang dikoreksi: {df_koreksi['persen_diubah'].mean():.2f}%")

file_id  kata_asr  kata_acuan  kata_diubah  persen_diubah
 APS-01      1320        1323            8           0.60
 APS-02      1388        1388            1           0.07
 APS-03      1138        1138            3           0.26
 APS-04      1519        1515           30           1.98
 APS-05      1022        1021            3           0.29
 APS-06      2513        2513            6           0.24
 IMK-01      1923        1923            1           0.05
 IMK-02      2293        2291            1           0.04
 IMK-03      1872        1871            4           0.21
 IMK-04      1951        1951            9           0.46
 IMK-05      2358        2355           35           1.49
 AOK-01      1338        1361           35           2.57
 AOK-02      2643        2667          113           4.24
 AOK-03      1095        1096           20           1.82
 AOK-04      1731        1736           42           2.42
 AOK-05      1533        1541           23           1.49
 AOK-06      1

## 10. Glosarium istilah teknis (subbab 3.3.3)

Satu baris per **bentuk permukaan**, sehingga ketentuan nomor 2 terpenuhi: satu entri dapat memiliki beberapa bentuk, misalnya `database` dan `basis data`, dengan penilaian dijatuhkan pada bentuk yang diucapkan.

Kolom `tanggal_kunci` adalah bukti kepatuhan pada ketentuan nomor 3, bahwa susunan entri dikunci sebelum koreksi transkrip acuan rampung.

Glosarium disusun untuk tiga mata kuliah: APS, IMK, dan AOK.

In [ ]:
GLOSARIUM_PATH = OUT / "glosarium.csv"

if not GLOSARIUM_PATH.exists():
    contoh = [
        # mata_kuliah, entri_id, bentuk_kanonik, bentuk_permukaan, sumber, tanggal_kunci
        ("APS", "APS-001", "use case diagram", "use case diagram", "buku pedoman kurikulum 2024", ""),
        ("APS", "APS-001", "use case diagram", "diagram use case",  "materi ajar", ""),
        ("IMK", "IMK-001", "usability",        "usability",         "buku pedoman kurikulum 2024", ""),
        ("IMK", "IMK-001", "usability",        "kebergunaan",       "materi ajar", ""),
        ("AOK", "AOK-001", "pipeline",         "pipeline",          "buku pedoman kurikulum 2024", ""),
    ]
    pd.DataFrame(contoh, columns=["mata_kuliah", "entri_id", "bentuk_kanonik",
                                  "bentuk_permukaan", "sumber", "tanggal_kunci"]
                 ).to_csv(GLOSARIUM_PATH, index=False, encoding="utf-8-sig")
    print("Templat glosarium dibuat:", GLOSARIUM_PATH)
else:
    print("Glosarium sudah ada, tidak ditimpa.")

glosarium = pd.read_csv(GLOSARIUM_PATH, dtype=str, keep_default_na=False)
print(glosarium.groupby("mata_kuliah").agg(entri=("entri_id", "nunique"),
                                           bentuk=("bentuk_permukaan", "count")).to_string())

In [ ]:
# Pemeriksaan konsistensi ejaan istilah pada transkrip acuan.
# Lintasan kedua: tidak butuh audio, cocok dikerjakan setelah TSV jadi.
if master is not None:
    laporan = []
    for mk, g_glos in glosarium.groupby("mata_kuliah"):
        teks_mk = " ".join(master.loc[master["mata_kuliah"] == mk, "teks_acuan"].astype(str))
        teks_l = teks_mk.lower()
        for _, r in g_glos.iterrows():
            bentuk = r["bentuk_permukaan"].strip()
            if not bentuk:
                continue
            n = len(re.findall(r"(?<!\w)" + re.escape(bentuk.lower()) + r"(?!\w)", teks_l))
            varian = sorted(set(re.findall(r"(?<!\w)" + re.escape(bentuk) + r"(?!\w)",
                                           teks_mk, flags=re.I)))
            laporan.append({"mata_kuliah": mk, "entri_id": r["entri_id"],
                            "bentuk_permukaan": bentuk, "kemunculan": n,
                            "varian_kapitalisasi": " | ".join(varian[:6]),
                            "jumlah_varian": len(varian)})

    df_glos = (pd.DataFrame(laporan)
                 .sort_values(["mata_kuliah", "kemunculan"], ascending=[True, False]))
    df_glos.to_csv(OUT / "cek_glosarium.csv", index=False, encoding="utf-8-sig")

    nihil = df_glos[df_glos["kemunculan"] == 0]
    print(f"Entri yang tidak pernah muncul di transkrip acuan: {len(nihil)}")
    if len(nihil):
        print(nihil[["mata_kuliah", "bentuk_permukaan"]].to_string(index=False))

    tidak_seragam = df_glos[df_glos["jumlah_varian"] > 1]
    print(f"\nEntri dengan kapitalisasi tidak seragam: {len(tidak_seragam)}")
    if len(tidak_seragam):
        print(tidak_seragam[["mata_kuliah", "bentuk_permukaan",
                             "varian_kapitalisasi"]].to_string(index=False))

## 11. Ringkasan untuk Bab 4

Jumlah rekaman, durasi, dan sebarannya per mata kuliah, sebagaimana dijanjikan pada akhir subbab 3.3.1.

In [ ]:
if master is not None:
    m = manifest.copy()
    for k in ("durasi_asli_detik", "durasi_potong_detik", "trim_awal_detik", "trim_akhir_detik"):
        m[k] = pd.to_numeric(m[k], errors="coerce")
    m["dibuang_detik"] = m["trim_awal_detik"].fillna(0) + m["trim_akhir_detik"].fillna(0)

    rk = (m.groupby(["mata_kuliah", "nama_mata_kuliah"])
            .agg(jumlah_rekaman=("file_id", "count"),
                 total_detik=("durasi_potong_detik", "sum"),
                 rerata_detik=("durasi_potong_detik", "mean"),
                 dibuang_detik=("dibuang_detik", "sum"))
            .reset_index())
    rk["total_jam"]    = (rk["total_detik"] / 3600).round(2)
    rk["rerata_menit"] = (rk["rerata_detik"] / 60).round(1)

    seg = (master.assign(dur=lambda d: d["dur"].astype(float),
                         n_kata=lambda d: d["teks_acuan"].astype(str).str.split().str.len())
                 .groupby("mata_kuliah")
                 .agg(jumlah_segmen=("seg_id", "count"),
                      rerata_dur_segmen=("dur", "mean"),
                      jumlah_kata=("n_kata", "sum")))
    seg["rerata_dur_segmen"] = seg["rerata_dur_segmen"].round(2)

    tabel = rk.merge(seg, on="mata_kuliah")[
        ["mata_kuliah", "nama_mata_kuliah", "jumlah_rekaman", "total_jam",
         "rerata_menit", "dibuang_detik", "jumlah_segmen",
         "rerata_dur_segmen", "jumlah_kata"]]
    tabel.to_csv(OUT / "ringkasan_data.csv", index=False, encoding="utf-8-sig")
    print(tabel.to_string(index=False))
    print(f"\nTOTAL: {tabel['jumlah_rekaman'].sum()} rekaman, "
          f"{tabel['total_jam'].sum():.2f} jam, "
          f"{tabel['jumlah_segmen'].sum()} segmen, "
          f"{tabel['jumlah_kata'].sum()} kata, "
          f"{tabel['dibuang_detik'].sum():.0f} detik dibuang sebagai bukan-materi")

In [ ]:
# Metadata run. Struktur bertingkat, sehingga disimpan sebagai JSON, bukan CSV.
import platform, sys
meta = {
    "tanggal": datetime.datetime.now().isoformat(timespec="seconds"),
    "tahap": "penyusunan transkrip acuan",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "konfigurasi": CFG,
    "ambang_prioritas_simak": AMBANG,
    "ambang_intro": AMBANG_INTRO,
    "versi_pustaka": {},
}
for nama in ["faster_whisper", "ctranslate2", "pandas", "numpy", "onnxruntime"]:
    try:
        meta["versi_pustaka"][nama] = __import__(nama).__version__
    except Exception:
        meta["versi_pustaka"][nama] = None

(OUT / "run_meta_transkrip_acuan.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(meta["versi_pustaka"], indent=2))

---

## Keluaran

| Berkas | Isi |
|---|---|
| `data/raw/audio_src/*.m4a` | audio unduhan, apa adanya |
| `data/raw/json/*.json` | keluaran large-v3, arsip, tidak disunting |
| `data/processed/audio/*.wav` | 16 kHz mono PCM 16-bit, ternormalisasi, terpotong |
| `data/processed/srt/*.srt` | bahan koreksi |
| `data/processed/srt_done/*.srt` | hasil koreksi manual |
| `data/output/master/*.tsv` | **master transkrip acuan per rekaman** |
| `data/output/transkrip_acuan_gabungan.tsv` | master gabungan |
| `data/output/manifest_rekaman.csv` | daftar rekaman, durasi, trim, penyaringan, split |
| `data/output/glosarium.csv` | glosarium per mata kuliah |
| `data/output/review_priority.csv` | segmen kandidat halusinasi |
| `data/output/laporan_validasi.csv` | hasil pemeriksaan struktural |
| `data/output/ringkasan_koreksi.csv` | proporsi kata yang dikoreksi |
| `data/output/cek_glosarium.csv` | konsistensi ejaan istilah |
| `data/output/ringkasan_data.csv` | tabel untuk Bab 4 |
| `data/output/run_meta_transkrip_acuan.json` | versi pustaka dan parameter |

## Membuka TSV di Excel tanpa merusak data

Jangan klik ganda. Gunakan `Data → From Text/CSV`, pilih **UTF-8**, pemisah **Tab**, lalu pada langkah pratinjau tetapkan seluruh kolom sebagai **Text**. Sebelum menempel apa pun, matikan `File → Options → Proofing → AutoCorrect Options`: kapitalisasi otomatis akan mengubah `eee` menjadi `Eee` dan melanggar pedoman verbatim.

Bila menyunting lewat Excel, simpan kembali sebagai **Unicode Text (.txt)** lalu ganti namanya, atau lebih aman sunting lewat pandas.

## Catatan yang layak masuk Bab 4

1. **Intro menggeser stempel waktu.** Kedelapan rekaman AOK memakai intro seragam 25 detik, dan Whisper menempelkan segmen pertamanya mundur sampai detik 1,5 sehingga teks salam menerima stempel waktu sekitar 23 detik lebih awal dari ujaran sebenarnya. APS dan IMK yang tanpa intro tidak menunjukkan gejala itu. Pasangan ini berfungsi sebagai pembanding alami dan menjadi bukti bahwa praproses audio pada 3.6.1 bukan sekadar penyeragaman format.
2. **Transkrip acuan diturunkan dari large-v3**, yang juga menjadi konfigurasi T1-E. Kuhn, et al. (2024) menempuh cara yang sama, tetapi sisa kesalahan yang lolos koreksi akan menguntungkan T1-E. `ringkasan_koreksi.csv` menyediakan angka proporsi koreksi sebagai keterangan mutu.
3. **Label APS memuat lebih dari satu penutur.** Kolom `penutur` mencatatnya, dan itu berkaitan dengan ketentuan 3.6.2 nomor 4 bahwa perolehan Tahap 3 berlaku sebagai adaptasi domain, bukan generalisasi lintas penutur.
4. **Penyaringan bersandar pada bahan kajian, bukan judul rekaman.** Pengajar pada APS-02 menyebut mata kuliah Rekayasa Perangkat Lunak, tetapi materinya beririsan dengan Analisis dan Perancangan Sistem dan memenuhi kriteria penyaringan nomor 1. Alasannya tercatat pada kolom `catatan` di manifest.

## Langkah berikutnya

Master ini menjadi masukan bagi tahap yang belum dicakup notebook ini: pembagian data 65 : 10 : 25 distratifikasi per mata kuliah (3.6.2), praproses teks dan penandaan glosarium (3.6.3), lalu Tahap 1 sampai Tahap 3 (3.6.5–3.6.7).